In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../')

# Demo for Self-Forcing - Multi-Prompt

This notebook demonstrates video generation with multiple prompts using the Self-Forcing model.


In [ ]:
"""
Demo for Self-Forcing.
"""

import os
import re
import random
import time
import base64
import argparse
import hashlib
import shutil
import subprocess
import urllib.request
from io import BytesIO
from PIL import Image
import numpy as np
import torch
from omegaconf import OmegaConf
import queue
from threading import Thread, Event

from pipeline import CausalInferencePipeline
from demo_utils.constant import ZERO_VAE_CACHE
from demo_utils.vae_block3 import VAEDecoderWrapper
from utils.wan_wrapper import WanDiffusionWrapper, WanTextEncoder
from demo_utils.utils import generate_timestamp
from demo_utils.memory import gpu, get_cuda_free_memory_gb, DynamicSwapInstaller, move_model_to_device_with_memory_preservation
from copy import deepcopy

debug = True


## Configuration

Set the checkpoint path, config path, and other parameters here.


In [ ]:
# Configuration
workdir = '../'
checkpoint_path = os.path.join(workdir, 'checkpoints/self_forcing_dmd.pt')
config_path = os.path.join(workdir, 'configs/self_forcing_dmd.yaml')
use_trt = False  # Set to True to use TensorRT

print(f'Free VRAM {get_cuda_free_memory_gb(gpu)} GB')
low_memory = get_cuda_free_memory_gb(gpu) < 80


## Global Variables

Initialize global variables for dynamic model switching and frame tracking.


In [ ]:
# Global variables for dynamic model switching
current_vae_decoder = None
current_use_taehv = True
fp8_applied = False
torch_compile_applied = True
global frame_number
frame_number = 0
anim_name = ""
frame_rate = 12
num_blocks = 7

## VAE Decoder Initialization Function


In [ ]:
def initialize_vae_decoder(use_taehv=False, use_trt=False):
    """Initialize VAE decoder based on the selected option"""
    global current_vae_decoder, current_use_taehv

    if use_trt:
        from demo_utils.vae import VAETRTWrapper
        current_vae_decoder = VAETRTWrapper()
        return current_vae_decoder

    if use_taehv:
        from demo_utils.taehv import TAEHV
        # Check if taew2_1.pth exists in checkpoints folder, download if missing
        taehv_checkpoint_path = os.path.join(workdir, "checkpoints/taew2_1.pth")
        if not os.path.exists(taehv_checkpoint_path):
            print(f"taew2_1.pth not found in checkpoints folder {taehv_checkpoint_path}. Downloading...")
            os.makedirs("checkpoints", exist_ok=True)
            download_url = "https://github.com/madebyollin/taehv/raw/main/taew2_1.pth"
            try:
                urllib.request.urlretrieve(download_url, taehv_checkpoint_path)
                print(f"Successfully downloaded taew2_1.pth to {taehv_checkpoint_path}")
            except Exception as e:
                print(f"Failed to download taew2_1.pth: {e}")
                raise

        class DotDict(dict):
            __getattr__ = dict.__getitem__
            __setattr__ = dict.__setitem__

        class TAEHVDiffusersWrapper(torch.nn.Module):
            def __init__(self):
                super().__init__()
                self.dtype = torch.float16
                self.taehv = TAEHV(checkpoint_path=taehv_checkpoint_path).to(self.dtype)
                self.config = DotDict(scaling_factor=1.0)

            def decode(self, latents, return_dict=None):
                # n, c, t, h, w = latents.shape
                # low-memory, set parallel=True for faster + higher memory
                return self.taehv.decode_video(latents, parallel=False).mul_(2).sub_(1)

        current_vae_decoder = TAEHVDiffusersWrapper()
    else:
        current_vae_decoder = VAEDecoderWrapper()
        vae_state_dict = torch.load(os.path.join(workdir, 'wan_models/Wan2.1-T2V-1.3B/Wan2.1_VAE.pth'), map_location="cpu")
        decoder_state_dict = {}
        for key, value in vae_state_dict.items():
            if 'decoder.' in key or 'conv2' in key:
                decoder_state_dict[key] = value
        current_vae_decoder.load_state_dict(decoder_state_dict)

    current_vae_decoder.eval()
    current_vae_decoder.to(dtype=torch.float16)
    current_vae_decoder.requires_grad_(False)
    current_vae_decoder.to(gpu)
    current_use_taehv = use_taehv

    print(f"✅ VAE decoder initialized with {'TAEHV' if use_taehv else 'default VAE'}")
    return current_vae_decoder


In [ ]:
# Load models
config = OmegaConf.load(config_path)
default_config = OmegaConf.load(os.path.join(workdir, "configs/default_config.yaml"))
config = OmegaConf.merge(default_config, config)

text_encoder = WanTextEncoder(workdir)

# Initialize with default VAE
vae_decoder = initialize_vae_decoder(use_taehv=False, use_trt=use_trt)

transformer = WanDiffusionWrapper(is_causal=True, dir=workdir, num_blocks=num_blocks)
state_dict = torch.load(checkpoint_path, map_location="cpu")
transformer.load_state_dict(state_dict['generator_ema'])

text_encoder.eval()
transformer.eval()

transformer.to(dtype=torch.float16)
text_encoder.to(dtype=torch.bfloat16)

text_encoder.requires_grad_(False)
transformer.requires_grad_(False)

pipeline = CausalInferencePipeline(
    config,
    device=gpu,
    generator=transformer,
    text_encoder=text_encoder,
    vae=vae_decoder,
)

if low_memory:
    DynamicSwapInstaller.install_model(text_encoder, device=gpu)
else:
    text_encoder.to(gpu)
transformer.to(gpu)

models_compiled = False


In [ ]:
transformer

## Helper Functions


In [ ]:
def tensor_to_base64_frame(frame_tensor):
    """Convert a single frame tensor to base64 image string."""
    global frame_number, anim_name
    # Clamp and normalize to 0-255
    frame = torch.clamp(frame_tensor.float(), -1., 1.) * 127.5 + 127.5
    frame = frame.to(torch.uint8).cpu().numpy()

    # CHW -> HWC
    if len(frame.shape) == 3:
        frame = np.transpose(frame, (1, 2, 0))

    # Convert to PIL Image
    if frame.shape[2] == 3:  # RGB
        image = Image.fromarray(frame, 'RGB')
    else:  # Handle other formats
        image = Image.fromarray(frame)

    # Convert to base64
    buffer = BytesIO()
    image.save(buffer, format='JPEG', quality=100)
    if not os.path.exists(os.path.join(workdir, "./images/%s" % anim_name)):
        os.makedirs(os.path.join(workdir, "./images/%s" % anim_name))
    frame_number += 1
    image.save(os.path.join(workdir, "./images/%s/%s_%03d.jpg" % (anim_name, anim_name, frame_number)))
    img_str = base64.b64encode(buffer.getvalue()).decode()
    return f"data:image/jpeg;base64,{img_str}"


def generate_mp4_from_images(image_directory, output_video_path, fps=24):
    """
    Generate an MP4 video from a directory of images ordered alphabetically.

    :param image_directory: Path to the directory containing images.
    :param output_video_path: Path where the output MP4 will be saved.
    :param fps: Frames per second for the output video.
    """
    global anim_name, workdir
    
    # Create output directory if it doesn't exist
    output_dir = os.path.dirname(output_video_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
    
    # Construct full path to image directory
    image_subdir = os.path.join(image_directory, anim_name)
    if not os.path.exists(image_subdir):
        raise FileNotFoundError(f"Image directory not found: {image_subdir}")
    
    # Check if any images exist
    image_pattern = os.path.join(image_subdir, f"{anim_name}_%03d.jpg")
    # Try to find at least one matching file
    test_pattern = os.path.join(image_subdir, f"{anim_name}_001.jpg")
    if not os.path.exists(test_pattern):
        # Try alternative numbering
        import glob
        pattern = os.path.join(image_subdir, f"{anim_name}_*.jpg")
        files = sorted(glob.glob(pattern))
        if not files:
            raise FileNotFoundError(f"No images found in {image_subdir}")
        print(f"Found {len(files)} images in {image_subdir}")
    
    # Find ffmpeg executable
    ffmpeg_path = shutil.which('ffmpeg')
    if ffmpeg_path is None:
        # Fallback to known path if not in PATH
        known_paths = [
            '/share/u/wendler/bin/ffmpeg',
            '/usr/bin/ffmpeg',
            '/usr/local/bin/ffmpeg'
        ]
        for path in known_paths:
            if os.path.exists(path):
                ffmpeg_path = path
                break
        
        if ffmpeg_path is None:
            raise FileNotFoundError("ffmpeg not found. Please install ffmpeg or add it to your PATH.")
    
    # Construct the ffmpeg command with absolute paths
    cmd = [
        ffmpeg_path,
        '-y',  # Overwrite output file if it exists
        '-framerate', str(fps),
        '-i', image_pattern,
        '-c:v', 'libx264',
        '-pix_fmt', 'yuv420p',
        output_video_path
    ]
    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"Video saved to {output_video_path}")
    except subprocess.CalledProcessError as e:
        print(f"ffmpeg error: {e}")
        print(f"ffmpeg stderr: {e.stderr}")
        print(f"ffmpeg stdout: {e.stdout}")
        raise


def calculate_sha256(data):
    # Convert data to bytes if it's not already
    if isinstance(data, str):
        data = data.encode()
    # Calculate SHA-256 hash
    sha256_hash = hashlib.sha256(data).hexdigest()
    return sha256_hash


## Video Generation Function

This is the main function that generates videos with multiple prompts. The prompt format is:
- `prompt1|block_idx:prompt2|block_idx:prompt3` where block_idx specifies which block to apply the prompt.


In [ ]:
@torch.no_grad()
def generate_video_stream(prompt, seed, enable_torch_compile=False, enable_fp8=False, use_taehv=False, num_blocks=7):
    """Generate video and return frames."""
    global models_compiled, torch_compile_applied, fp8_applied, current_vae_decoder, current_use_taehv, frame_rate, anim_name, frame_number

    try:
        frame_number = 0
        
        # Handle VAE decoder switching
        if use_taehv != current_use_taehv:
            print(f"🔄 Switching VAE decoder to {'TAEHV' if use_taehv else 'default VAE'}")
            current_vae_decoder = initialize_vae_decoder(use_taehv=use_taehv)
            # Update pipeline with new VAE decoder
            pipeline.vae = current_vae_decoder

        # Handle FP8 quantization
        if enable_fp8 and not fp8_applied:
            print("🔧 Applying FP8 quantization to transformer")
            from torchao.quantization.quant_api import quantize_, Float8DynamicActivationFloat8WeightConfig, PerTensor
            quantize_(transformer, Float8DynamicActivationFloat8WeightConfig(granularity=PerTensor()))
            fp8_applied = True

        # Text encoding
        print("Encoding text prompt...")
        cond_dicts = {}
        prompts = prompt.split('|')
        for idx, prompt in enumerate(prompts):
            if idx > 0:
                block_idx = int(prompt.split(':')[0])
                prompt = prompt.split(':')[1]
            else:
                block_idx = 0
                prompt = prompt
            print(f"block_idx: {block_idx}, prompt: {prompt}")
            conditional_dict = text_encoder(text_prompts=[prompt])
            for key, value in conditional_dict.items():
                conditional_dict[key] = value.to(dtype=torch.float16)
            cond_dicts[block_idx] = conditional_dict
        print(cond_dicts)
        

        if low_memory:
            gpu_memory_preservation = get_cuda_free_memory_gb(gpu) + 5
            move_model_to_device_with_memory_preservation(
                text_encoder,target_device=gpu, preserved_memory_gb=gpu_memory_preservation)

        # Handle torch.compile if enabled
        torch_compile_applied = enable_torch_compile
        if enable_torch_compile and not models_compiled:
            # Compile transformer and decoder
            print("Compiling models (may take 5-10 minutes)...")
            transformer.compile(mode="max-autotune-no-cudagraphs")
            if not current_use_taehv and not low_memory and not use_trt:
                current_vae_decoder.compile(mode="max-autotune-no-cudagraphs")
            models_compiled = True

        # Initialize generation
        print("Initializing generation...")

        rnd = torch.Generator(gpu).manual_seed(seed)

        pipeline._initialize_kv_cache(batch_size=1, dtype=torch.float16, device=gpu)
        pipeline._initialize_crossattn_cache(batch_size=1, dtype=torch.float16, device=gpu)

        empty_kv_cache = []
        empty_crossattn_cache = []
        for kv in pipeline.kv_cache1:
            kv_copy = deepcopy(kv)
            kv_copy['k'] = torch.zeros_like(kv['k'], device=kv['k'].device, dtype=kv['k'].dtype)
            kv_copy['v'] = torch.zeros_like(kv['v'], device=kv['v'].device, dtype=kv['v'].dtype)
            empty_kv_cache.append(kv_copy)
        for crossattn in pipeline.crossattn_cache:
            crossattn_copy = deepcopy(crossattn)
            crossattn_copy['k'] = torch.zeros_like(crossattn['k'], device=crossattn['k'].device, dtype=crossattn['k'].dtype)
            crossattn_copy['v'] = torch.zeros_like(crossattn['v'], device=crossattn['v'].device, dtype=crossattn['v'].dtype)
            empty_crossattn_cache.append(crossattn_copy)

        # Generation parameters
        current_start_frame = 0
        num_input_frames = 0


        noise = torch.randn([1, 3*num_blocks, 16, 60, 104], device=gpu, dtype=torch.float16, generator=rnd)
        # batch x frames x vae channels x vae height x vae width
        # frames = 1 + 4 * 5 = 21 --> WAN-like VAE 
        # or: 21 / 7 = 3 --> 3 frames per block?

        all_num_frames = [pipeline.num_frame_per_block] * num_blocks
        if current_use_taehv:
            vae_cache = None
        else:
            vae_cache = ZERO_VAE_CACHE
            for i in range(len(vae_cache)):
                vae_cache[i] = vae_cache[i].to(device=gpu, dtype=torch.float16)

        total_frames_sent = 0
        generation_start_time = time.time()

        print("Generating frames...")
        print(f"🔄 cond_dicts: {list(cond_dicts.keys())}")
        skip_cache = False
        last_cond_idx = 0
        for idx, current_num_frames in enumerate(all_num_frames):
            if idx in cond_dicts:
                conditional_dict = cond_dicts[idx]
                last_cond_idx = idx
                if idx > 0:
                    skip_cache = True
            else:
                conditional_dict = cond_dicts[last_cond_idx]
                skip_cache = False
            
            print(f"last_cond_idx: {last_cond_idx}")
            

            print(f"🔄 Processing block {idx+1}/{len(all_num_frames)}")

            block_start_time = time.time()

            noisy_input = noise[:, current_start_frame -
                                num_input_frames:current_start_frame + current_num_frames - num_input_frames]

            # Denoising loop
            denoising_start = time.time()
            for index, current_timestep in enumerate(pipeline.denoising_step_list):

                timestep = torch.ones([1, current_num_frames], device=noise.device,
                                      dtype=torch.int64) * current_timestep

                print(f"noisy_input shape: {noisy_input.shape}")
                print(f"timestep.shape: {timestep.shape}")
                print(f"conditional_dict['prompt_embeds'] shape: {conditional_dict['prompt_embeds'].shape}")
                print(f"kv_cache1 length: {len(pipeline.kv_cache1)}")
                print(f"kv_cache1[0]['k'].shape: {pipeline.kv_cache1[0]['k'].shape}, kv_cache1[0]['v'].shape: {pipeline.kv_cache1[0]['v'].shape}")
                print(f"crossattn_cache length: {len(pipeline.crossattn_cache)}")
                print(f"crossattn_cache[0]['k'].shape: {pipeline.crossattn_cache[0]['k'].shape}, crossattn_cache[0]['v'].shape: {pipeline.crossattn_cache[0]['v'].shape}")

                if skip_cache:
                    kv_cache = empty_kv_cache
                    crossattn_cache = empty_crossattn_cache
                else:
                    kv_cache = pipeline.kv_cache1
                    crossattn_cache = pipeline.crossattn_cache
                print(f"kv_cache norms: {[kv['k'].norm().item() for kv in kv_cache]}")
                print(f"crossattn_cache norms: {[kv['k'].norm().item() for kv in crossattn_cache]}")
                
                if index < len(pipeline.denoising_step_list) - 1:
                    _, denoised_pred = transformer(
                        noisy_image_or_video=noisy_input,
                        conditional_dict=conditional_dict,
                        timestep=timestep,
                        kv_cache=kv_cache,
                        crossattn_cache=crossattn_cache,
                        current_start=current_start_frame * pipeline.frame_seq_length
                    )
                    next_timestep = pipeline.denoising_step_list[index + 1]
                    noisy_input = pipeline.scheduler.add_noise(
                        denoised_pred.flatten(0, 1),
                        torch.randn_like(denoised_pred.flatten(0, 1)),
                        next_timestep * torch.ones([1 * current_num_frames], device=noise.device, dtype=torch.long)
                    ).unflatten(0, denoised_pred.shape[:2])
                else:
                    _, denoised_pred = transformer(
                        noisy_image_or_video=noisy_input,
                        conditional_dict=conditional_dict,
                        timestep=timestep,
                        kv_cache=kv_cache,
                        crossattn_cache=crossattn_cache,
                        current_start=current_start_frame * pipeline.frame_seq_length
                    )

            denoising_time = time.time() - denoising_start
            print(f"⚡ Block {idx+1} denoising completed in {denoising_time:.2f}s")

            # Update KV cache for next block
            if idx != len(all_num_frames) - 1:
                transformer(
                    noisy_image_or_video=denoised_pred,
                    conditional_dict=conditional_dict,
                    timestep=torch.zeros_like(timestep),
                    kv_cache=pipeline.kv_cache1,
                    crossattn_cache=pipeline.crossattn_cache,
                    current_start=current_start_frame * pipeline.frame_seq_length,
                )


            # Decode to pixels and save frames
            print(f"🎨 Decoding block {idx+1} to pixels...")
            decode_start = time.time()
            if use_trt:
                all_current_pixels = []
                for i in range(denoised_pred.shape[1]):
                    is_first_frame = torch.tensor(1.0).cuda().half() if idx == 0 and i == 0 else \
                        torch.tensor(0.0).cuda().half()
                    outputs = vae_decoder.forward(denoised_pred[:, i:i + 1, :, :, :].half(), is_first_frame, *vae_cache)
                    # outputs = vae_decoder.forward(denoised_pred.float(), *vae_cache)
                    current_pixels, vae_cache = outputs[0], outputs[1:]
                    print(current_pixels.max(), current_pixels.min())
                    all_current_pixels.append(current_pixels.clone())
                pixels = torch.cat(all_current_pixels, dim=1)
                if idx == 0:
                    pixels = pixels[:, 3:, :, :, :]  # Skip first 3 frames of first block
            else:
                if current_use_taehv:
                    if vae_cache is None:
                        vae_cache = denoised_pred
                    else:
                        denoised_pred = torch.cat([vae_cache, denoised_pred], dim=1)
                        vae_cache = denoised_pred[:, -3:, :, :, :]
                    pixels = current_vae_decoder.decode(denoised_pred)
                    print(f"denoised_pred shape: {denoised_pred.shape}")
                    print(f"pixels shape: {pixels.shape}")
                    if idx == 0:
                        pixels = pixels[:, 3:, :, :, :]  # Skip first 3 frames of first block
                    else:
                        pixels = pixels[:, 12:, :, :, :]

                else:
                    pixels, vae_cache = current_vae_decoder(denoised_pred.half(), *vae_cache)
                    if idx == 0:
                        pixels = pixels[:, 3:, :, :, :]  # Skip first 3 frames of first block

            decode_time = time.time() - decode_start
            print(f"🎨 Block {idx+1} VAE decoding completed in {decode_time:.2f}s")

            # Save frames
            block_frames = pixels.shape[1]
            print(f"💾 Saving {block_frames} frames from block {idx+1}...")

            for frame_idx in range(block_frames):
                frame_tensor = pixels[0, frame_idx].cpu()
                tensor_to_base64_frame(frame_tensor)
                total_frames_sent += 1

            block_time = time.time() - block_start_time
            print(f"✅ Block {idx+1} completed in {block_time:.2f}s ({block_frames} frames saved)")

            current_start_frame += current_num_frames

        generation_time = time.time() - generation_start_time
        print(f"🎉 Generation completed in {generation_time:.2f}s! {total_frames_sent} frames saved")

        generate_mp4_from_images(os.path.join(workdir, "./images"),os.path.join(workdir, "./videos/"+anim_name+".mp4"), frame_rate )
        print(f"✅ Video saved to {os.path.join(workdir, f'./videos/{anim_name}.mp4')}")
        return os.path.join(workdir, f'./videos/{anim_name}.mp4')

    except Exception as e:
        if debug:
            raise e
        print(f"❌ Generation failed: {e}")
        raise


## Example Usage

Generate a video with multiple prompts. The format is:
- `prompt1|block_idx:prompt2|block_idx:prompt3` where block_idx specifies which block to apply the prompt.
- For example: `A cat walking|3:A dog running|7:A bird flying` will use "A cat walking" for blocks 0-2, "A dog running" for blocks 3-6, and "A bird flying" for blocks 7-13.


In [ ]:
# Set your prompt here
prompt = "An astronaut riding a pig in space."  # Or use multi-prompt format like "prompt1|3:prompt2|7:prompt3"

# Generate seed (use -1 for random, or set a specific seed)
seed = -1
if seed == -1:
    seed = random.randint(0, 2**32)

# Extract words up to the first punctuation or newline
words_up_to_punctuation = re.split(r'[^\w\s]', prompt)[0].strip() if prompt else ''
if not words_up_to_punctuation:
    words_up_to_punctuation = re.split(r'[\n\r]', prompt)[0].strip()

# Calculate SHA-256 hash of the entire prompt
sha256_hash = calculate_sha256(prompt)

# Create anim_name with the extracted words and first 10 characters of the hash
anim_name = f"{words_up_to_punctuation[:20]}_{str(seed)}_{sha256_hash[:10]}"

print(f"Starting generation with prompt: {prompt}")
print(f"Seed: {seed}")
print(f"Animation name: {anim_name}")
# Generate the video
mp4_path = generate_video_stream(prompt, seed, torch_compile_applied, fp8_applied, current_use_taehv)
import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open(mp4_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width="640" height="480" controls>
  <source src="{data_url}" type="video/mp4">
  Your browser does not support the video tag.
</video>
""")
